In [1]:
import os

# Dataset
from sklearn.datasets import fetch_california_housing

# Data manipulation
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

# Data preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Regression models
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb

# Evaluation metrics
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

In [2]:
# Results directory (relative to project root)
RESULTS_DIR = "../../results/regression"
os.makedirs(RESULTS_DIR, exist_ok=True)

RESULTS_FILE = os.path.join(RESULTS_DIR, "regression_metrics.csv")


def save_model_results(model_name, y_true, y_pred, dataset="test"):

    results = {
        "Model": model_name,
        "Dataset": dataset,
        "MSE": mean_squared_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "MAE": mean_absolute_error(y_true, y_pred),
        "R2": r2_score(y_true, y_pred)
    }

    df = pd.DataFrame([results])

    df.to_csv(
        RESULTS_FILE,
        mode="a",
        header=not os.path.exists(RESULTS_FILE),
        index=False
    )

    print(f"{model_name} regression metrics saved successfully!")

In [3]:
# Fetch dataset
housing = fetch_california_housing(as_frame=True)

# Features and target
X = housing.data
y = housing.target

In [4]:
# First split: 70% training, 30% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.40,
    random_state=42
)

# Second split: 15% validation, 15% testing
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42
)

In [5]:
print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Testing:", X_test.shape, y_test.shape)

Training: (12384, 8) (12384,)
Validation: (4128, 8) (4128,)
Testing: (4128, 8) (4128,)


In [6]:
# Model 1: Default configuration with moderate regularisation
model1 = Lasso(alpha=0.01, max_iter=1000)
model1.fit(X_train, y_train)

# Model 2: Stronger regularisation
model2 = Lasso(alpha=0.1, max_iter=2000)
model2.fit(X_train, y_train)

# Model 3: Higher regularisation
model3 = Lasso(alpha=0.5, max_iter=5000)
model3.fit(X_train, y_train)

# Model 4: Very strong regularisation
model4 = Lasso(alpha=1.0, max_iter=5000)
model4.fit(X_train, y_train)

# Model 5: Lower convergence tolerance
model5 = Lasso(alpha=0.05, max_iter=5000, tol=0.00001)
model5.fit(X_train, y_train)

# Model 6: Random coordinate selection
model6 = Lasso(alpha=0.1, max_iter=5000, selection="random", random_state=42)
model6.fit(X_train, y_train)

# Model 7: Cyclic coordinate selection
model7 = Lasso(alpha=0.1, max_iter=5000, selection="cyclic")
model7.fit(X_train, y_train)

# Model 8: Positive coefficients only
model8 = Lasso(alpha=0.1, max_iter=5000, positive=True)
model8.fit(X_train, y_train)

# Model 9: Warm start enabled
model9 = Lasso(alpha=0.1, max_iter=5000, warm_start=True)
model9.fit(X_train, y_train)

# Model 10: No intercept
model10 = Lasso(alpha=0.1, max_iter=5000, fit_intercept=False)
model10.fit(X_train, y_train)

,"alpha alpha: float, default=1.0Constant that multiplies the L1 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Lasso` object is not advised.Instead, you should use the :class:`LinearRegression` object.",0.1
,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",False
,"max_iter max_iter: int, default=1000The maximum number of iterations.",5000
,"precompute precompute: bool or array-like of shape (n_features, n_features), default=FalseWhether to use a precomputed Gram matrix to speed upcalculations. The Gram matrix can also be passed as argument.For sparse input this option is always ``False`` to preserve sparsity.",False
,"copy_X copy_X: bool, default=TrueIf ``True``, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-4The tolerance for the optimization: if the updates are smaller or equal to``tol``, the optimization code checks the dual gap for optimality and continuesuntil it is smaller or equal to ``tol``, see Notes below.",0.0001
,"warm_start warm_start: bool, default=FalseWhen set to ``True``, reuse the solution of the previous call to fit asinitialization, otherwise, just erase the previous solution.See :term:`the Glossary <warm_start>`.",False
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.",False
,"random_state random_state: int, RandomState instance, default=NoneThe seed of the pseudo random number generator that selects a randomfeature to update. Used when ``selection`` == 'random'.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
,"selection selection: {'cyclic', 'random'}, default='cyclic'If set to 'random', a random coefficient is updated every iterationrather than looping over features sequentially by default. This(setting to 'random') often leads to significantly faster convergenceespecially when tol is higher than 1e-4.",'cyclic'
Name,Type,Value


In [7]:
lasso_models = {
    "Lasso_(alpha=0.01, max_iter=1000) ": model1,
    "Lasso_(alpha=0.1, max_iter=2000)": model2,
    "Lasso_(alpha=0.5, max_iter=5000)": model3,
    "Lasso_(alpha=1.0, max_iter=5000)": model4,
    "Lasso_alpha=0.05, max_iter=5000, tol=0.00001": model5,
    "Lasso_(alpha=0.1, max_iter=5000, selection=random, random_state=42)": model6,
    "Lasso_(alpha=0.1, max_iter=5000, selection=cyclic)": model7,
    "Lasso_(alpha=0.1, max_iter=5000, positive=True)": model8,
    "Lasso_(alpha=0.1, max_iter=5000, warm_start=True)": model9,
    "Lasso_(alpha=0.1, max_iter=5000, fit_intercept=False)": model10
}

In [8]:
def generate_predictions(models, X):
    return {
        name: model.predict(X)
        for name, model in models.items()
    }


y_val_predictions = generate_predictions(lasso_models, X_val)
y_test_predictions = generate_predictions(lasso_models, X_test)

In [9]:
def evaluate_models(models, y_true, predictions, dataset="validation"):

    results = []

    for name, model in models.items():
        y_pred = predictions[name]

        results.append({
            "Model": name,
            "Dataset": dataset,
            "MSE": mean_squared_error(y_true, y_pred),
            "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
            "MAE": mean_absolute_error(y_true, y_pred),
            "R2": r2_score(y_true, y_pred)
        })

    return pd.DataFrame(results)

In [10]:
val_results = evaluate_models(
    lasso_models, y_val, y_val_predictions, "validation"
)

test_results = evaluate_models(
    lasso_models, y_test, y_test_predictions, "test"
)

print("VALIDATION RESULTS")
display(val_results)

print("TEST RESULTS")
display(test_results)

VALIDATION RESULTS


,Model,Dataset,MSE,RMSE,MAE,R2
0,"Lasso_(alpha=0.01, max_iter=1000)",validation,0.534857,0.731339,0.537953,0.590254
1,"Lasso_(alpha=0.1, max_iter=2000)",validation,0.611734,0.782135,0.584460,0.531359
2,"Lasso_(alpha=0.5, max_iter=5000)",validation,0.728507,0.853526,0.658310,0.441901
3,"Lasso_(alpha=1.0, max_iter=5000)",validation,0.944084,0.971640,0.765511,0.276750
4,"Lasso_alpha=0.05, max_iter=5000, tol=0.00001",validation,0.562589,0.750059,0.554907,0.569008
5,"Lasso_(alpha=0.1, max_iter=5000, selection=ran...",validation,0.611793,0.782172,0.584495,0.531314
6,"Lasso_(alpha=0.1, max_iter=5000, selection=cyc...",validation,0.611734,0.782135,0.584460,0.531359
7,"Lasso_(alpha=0.1, max_iter=5000, positive=True)",validation,0.660907,0.812962,0.612134,0.493688
8,"Lasso_(alpha=0.1, max_iter=5000, warm_start=True)",validation,0.611734,0.782135,0.584460,0.531359
9,"Lasso_(alpha=0.1, max_iter=5000, fit_intercept...",validation,0.646224,0.803881,0.604803,0.504937


TEST RESULTS


,Model,Dataset,MSE,RMSE,MAE,R2
0,"Lasso_(alpha=0.01, max_iter=1000)",test,0.541295,0.735728,0.536813,0.605218
1,"Lasso_(alpha=0.1, max_iter=2000)",test,0.611462,0.781960,0.580070,0.554044
2,"Lasso_(alpha=0.5, max_iter=5000)",test,0.736349,0.858108,0.657875,0.462960
3,"Lasso_(alpha=1.0, max_iter=5000)",test,0.973605,0.986714,0.775058,0.289923
4,"Lasso_alpha=0.05, max_iter=5000, tol=0.00001",test,0.562285,0.749857,0.552030,0.589910
5,"Lasso_(alpha=0.1, max_iter=5000, selection=ran...",test,0.611521,0.781998,0.580104,0.554001
6,"Lasso_(alpha=0.1, max_iter=5000, selection=cyc...",test,0.611462,0.781960,0.580070,0.554044
7,"Lasso_(alpha=0.1, max_iter=5000, positive=True)",test,0.661591,0.813383,0.608034,0.517483
8,"Lasso_(alpha=0.1, max_iter=5000, warm_start=True)",test,0.611462,0.781960,0.580070,0.554044
9,"Lasso_(alpha=0.1, max_iter=5000, fit_intercept...",test,0.644490,0.802801,0.599414,0.529956


In [11]:
for name in lasso_models:

    save_model_results(
        name,
        y_val,
        y_val_predictions[name],
        dataset="validation"
    )

    save_model_results(
        name,
        y_test,
        y_test_predictions[name],
        dataset="test"
    )

Lasso_(alpha=0.01, max_iter=1000)  regression metrics saved successfully!
Lasso_(alpha=0.01, max_iter=1000)  regression metrics saved successfully!
Lasso_(alpha=0.1, max_iter=2000) regression metrics saved successfully!
Lasso_(alpha=0.1, max_iter=2000) regression metrics saved successfully!
Lasso_(alpha=0.5, max_iter=5000) regression metrics saved successfully!
Lasso_(alpha=0.5, max_iter=5000) regression metrics saved successfully!
Lasso_(alpha=1.0, max_iter=5000) regression metrics saved successfully!
Lasso_(alpha=1.0, max_iter=5000) regression metrics saved successfully!
Lasso_alpha=0.05, max_iter=5000, tol=0.00001 regression metrics saved successfully!
Lasso_alpha=0.05, max_iter=5000, tol=0.00001 regression metrics saved successfully!
Lasso_(alpha=0.1, max_iter=5000, selection=random, random_state=42) regression metrics saved successfully!
Lasso_(alpha=0.1, max_iter=5000, selection=random, random_state=42) regression metrics saved successfully!
Lasso_(alpha=0.1, max_iter=5000, selec

In [12]:
def extract_coefficients(models, feature_names=None):

    results = []

    for name, model in models.items():

        for i, coefficient in enumerate(model.coef_):

            results.append({
                "Model": name,
                "Feature": feature_names[i] if feature_names is not None else f"Feature_{i}",
                "Coefficient": coefficient
            })

    return pd.DataFrame(results)

In [13]:
coefficient_results = extract_coefficients(
    lasso_models,
    X_train.columns
)

display(coefficient_results)

,Model,Feature,Coefficient
0,"Lasso_(alpha=0.01, max_iter=1000)",MedInc,4.094829e-01
1,"Lasso_(alpha=0.01, max_iter=1000)",HouseAge,1.010703e-02
2,"Lasso_(alpha=0.01, max_iter=1000)",AveRooms,-5.188282e-02
3,"Lasso_(alpha=0.01, max_iter=1000)",AveBedrms,3.939575e-01
4,"Lasso_(alpha=0.01, max_iter=1000)",Population,5.815954e-07
...,...,...,...
75,"Lasso_(alpha=0.1, max_iter=5000, fit_intercept...",AveBedrms,0.000000e+00
76,"Lasso_(alpha=0.1, max_iter=5000, fit_intercept...",Population,2.238159e-05
77,"Lasso_(alpha=0.1, max_iter=5000, fit_intercept...",AveOccup,-3.063382e-03
78,"Lasso_(alpha=0.1, max_iter=5000, fit_intercept...",Latitude,-4.095657e-02


In [14]:
intercept_results = pd.DataFrame([
    {
        "Model": name,
        "Intercept": model.intercept_
    }
    for name, model in lasso_models.items()
])

display(intercept_results)

,Model,Intercept
0,"Lasso_(alpha=0.01, max_iter=1000)",-35.060214
1,"Lasso_(alpha=0.1, max_iter=2000)",-7.837773
2,"Lasso_(alpha=0.5, max_iter=5000)",0.588065
3,"Lasso_(alpha=1.0, max_iter=5000)",1.350097
4,"Lasso_alpha=0.05, max_iter=5000, tol=0.00001",-22.695648
5,"Lasso_(alpha=0.1, max_iter=5000, selection=ran...",-7.824721
6,"Lasso_(alpha=0.1, max_iter=5000, selection=cyc...",-7.837773
7,"Lasso_(alpha=0.1, max_iter=5000, positive=True)",-0.024448
8,"Lasso_(alpha=0.1, max_iter=5000, warm_start=True)",-7.837773
9,"Lasso_(alpha=0.1, max_iter=5000, fit_intercept...",0.000000


In [15]:
all_results = pd.concat(
    [val_results, test_results],
    ignore_index=True
)

display(all_results)

,Model,Dataset,MSE,RMSE,MAE,R2
0,"Lasso_(alpha=0.01, max_iter=1000)",validation,0.534857,0.731339,0.537953,0.590254
1,"Lasso_(alpha=0.1, max_iter=2000)",validation,0.611734,0.782135,0.584460,0.531359
2,"Lasso_(alpha=0.5, max_iter=5000)",validation,0.728507,0.853526,0.658310,0.441901
3,"Lasso_(alpha=1.0, max_iter=5000)",validation,0.944084,0.971640,0.765511,0.276750
4,"Lasso_alpha=0.05, max_iter=5000, tol=0.00001",validation,0.562589,0.750059,0.554907,0.569008
5,"Lasso_(alpha=0.1, max_iter=5000, selection=ran...",validation,0.611793,0.782172,0.584495,0.531314
6,"Lasso_(alpha=0.1, max_iter=5000, selection=cyc...",validation,0.611734,0.782135,0.584460,0.531359
7,"Lasso_(alpha=0.1, max_iter=5000, positive=True)",validation,0.660907,0.812962,0.612134,0.493688
8,"Lasso_(alpha=0.1, max_iter=5000, warm_start=True)",validation,0.611734,0.782135,0.584460,0.531359
9,"Lasso_(alpha=0.1, max_iter=5000, fit_intercept...",validation,0.646224,0.803881,0.604803,0.504937


# Ridge Regression: Complete Study Notes

**Goal:** understand Ridge from first principles, know the important sklearn parameters, implement it on a real dataset, interpret the results, and prepare for interviews.

---

## 1. The problem Ridge solves

Linear regression predicts a continuous target as a weighted sum of features:

$$
\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_p x_p
$$

- $\hat{y}$: predicted target
- $\beta_0$: intercept
- $\beta_j$: coefficient for feature $x_j$
- $p$: number of features

Ordinary Least Squares (OLS) chooses coefficients that minimise the residual sum of squares:

$$
\text{RSS} = \sum_{i=1}^{n} (y_i - \hat{y}_i)^2
$$

OLS coefficients can become **large and unstable** when:

1. There are many features relative to observations (and if $p > n$, OLS has no unique solution).
2. Features are highly correlated (multicollinearity).
3. Predictors are noisy or redundant.

Unstable coefficients mean **high variance**: small changes in the training data produce very different models, which leads to overfitting.

---

## 2. What is Ridge Regression?

Ridge is linear regression with an **L2 penalty** on the coefficients:

$$
\min_{\beta_0, \boldsymbol{\beta}} \; \sum_{i=1}^{n} (y_i - \hat{y}_i)^2 \; + \; \alpha \sum_{j=1}^{p} \beta_j^2
$$

| Term | Meaning |
|---|---|
| $\sum_i (y_i - \hat{y}_i)^2$ | RSS: penalises prediction errors |
| $\alpha \sum_j \beta_j^2$ | L2 penalty: penalises large coefficients |
| $\alpha \ge 0$ | Regularisation strength |

- The **intercept $\beta_0$ is not penalised**. Penalising it would make predictions depend on where the target happens to be centred.
- Shrinkage is **not uniform**. How much each coefficient shrinks depends on the data, the feature correlations, and $\alpha$.

### Effect of alpha

| $\alpha$ | Behaviour |
|---|---|
| $0$ | No penalty; equivalent to OLS |
| Small | Mild shrinkage |
| Large | Strong shrinkage; coefficients move towards 0 |
| $\to \infty$ | All coefficients $\to 0$; predictions $\to$ mean of $y$ |

Ridge shrinks coefficients **towards** zero but generally **never exactly to zero**. It is not a feature-selection method; Lasso is.

### Bias–variance trade-off

| Regularisation | Bias | Variance | Typical result |
|---|---|---|---|
| Too little | Low | High | Overfits |
| Appropriate | Balanced | Balanced | Best generalisation |
| Too much | High | Low | Underfits |

The aim is not the smallest coefficients. It is the $\alpha$ that gives the best performance on **unseen** data.

---

## 3. Why feature scaling matters

The penalty acts on coefficient **magnitudes**, and magnitudes depend on units. Consider age (18–80) and income (10,000–150,000). Income's coefficient is naturally tiny, so it barely gets penalised, while age's coefficient gets penalised heavily. Unscaled Ridge penalises features unequally for arbitrary reasons.

The fix is to standardise each feature using the **training-set** mean and standard deviation:

$$
z = \frac{x - \mu_{\text{train}}}{\sigma_{\text{train}}}
$$

Always put the scaler inside a `Pipeline`. That way it is re-fitted on each training fold during cross-validation, and no information leaks from validation or test data.

---

## 4. `sklearn.linear_model.Ridge` — parameters

```python
from sklearn.linear_model import Ridge
model = Ridge(alpha=1.0)
```

| Parameter | Default | What it controls |
|---|---|---|
| `alpha` | `1.0` | Penalty strength. Can be an array of shape `(n_targets,)` for multi-output, giving one penalty per target. |
| `fit_intercept` | `True` | Whether to fit $\beta_0$. Only set `False` if the data is already centred. |
| `solver` | `'auto'` | Optimisation algorithm (see below). |
| `positive` | `False` | Forces coefficients $\ge 0$. Only the `'lbfgs'` solver supports this. |
| `max_iter` | `None` | Max iterations for iterative solvers (`sparse_cg`, `lsqr`, `sag`, `saga`, `lbfgs`). |
| `tol` | `1e-4` | Convergence tolerance for iterative solvers. |
| `random_state` | `None` | Used by `sag` / `saga` for shuffling. |
| `copy_X` | `True` | Leave as `True`. If `False`, `X` may be overwritten. |

**Fitted attributes:** `coef_`, `intercept_`, `n_iter_`, `n_features_in_`, `feature_names_in_`.

### Solvers

| Solver | Idea | Use when |
|---|---|---|
| `'auto'` | Picks based on data type and options | Default; almost always fine |
| `'svd'` | Singular value decomposition | Most numerically stable; dense only |
| `'cholesky'` | Closed-form via Cholesky decomposition | Fast for dense data with moderate $p$ |
| `'lsqr'` | Iterative least squares | Large or sparse data; usually fastest |
| `'sparse_cg'` | Conjugate gradient | Large sparse data |
| `'sag'` / `'saga'` | Stochastic average gradient | Very large $n$; **requires scaled features** |
| `'lbfgs'` | Quasi-Newton | Required for `positive=True` |

### Related estimators worth knowing

| Estimator | Why it matters |
|---|---|
| `RidgeCV` | Built-in alpha search. With the default `cv=None`, it uses very fast efficient leave-one-out CV. |
| `RidgeClassifier` / `RidgeClassifierCV` | Ridge for classification: targets are encoded as $\pm 1$ and fit as regression. |
| `KernelRidge` | Ridge with the kernel trick for nonlinear relationships. |
| `SGDRegressor(penalty='l2')` | Ridge-like model fitted with SGD for huge or streaming data. |

---

## 5. Ridge vs OLS vs Lasso vs Elastic Net

| Property | OLS | Ridge (L2) | Lasso (L1) | Elastic Net |
|---|---|---|---|---|
| Penalty | None | $\alpha \sum_j \beta_j^2$ | $\alpha \sum_j \lvert \beta_j \rvert$ | Mix of L1 and L2 |
| Coefficients exactly 0? | No | No | Yes | Yes |
| Feature selection | No | No | Yes | Yes |
| Correlated features | Unstable | Spreads weight across the group | Tends to pick one arbitrarily | Selects groups |
| Closed-form solution | Yes | Yes | No | No |
| Works when $p > n$ | No unique solution | Yes | Yes (selects at most $n$ features) | Yes |

**Geometric intuition:** the L2 constraint region is a circle (or sphere), so the optimum rarely lands on an axis. The L1 region is a diamond with corners on the axes, so solutions often hit a corner and coefficients become exactly 0.

> **sklearn gotcha:** `Lasso` and `ElasticNet` minimise $\frac{1}{2n}\text{RSS} + \alpha \cdot \text{penalty}$, while `Ridge` uses plain $\text{RSS} + \alpha \cdot \text{penalty}$. The same `alpha` value is **not comparable** across these estimators, so tune each one separately.

---

## 6. Practical exercise: diabetes progression

**Task:** predict a quantitative measure of disease progression one year after baseline (442 patients, 10 features). Check whether Ridge improves generalisation over OLS.

> Note: `load_diabetes()` returns features that are **already mean-centred and scaled**. Use `load_diabetes(scaled=False)` for raw values. We still use `StandardScaler` in the pipeline because that is the correct workflow for real data.

### 6.1 Load data

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

data = load_diabetes(as_frame=True)
X, y = data.data, data.target

print("X shape:", X.shape)
print("Features:", list(X.columns))
X.head()
```

### 6.2 Metrics

$$
\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} \lvert y_i - \hat{y}_i \rvert
\qquad
\text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}
\qquad
R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}
$$

- **MAE:** average error in target units; robust to outliers.
- **RMSE:** also in target units; punishes large errors more.
- **$R^2$:** improvement over predicting the mean. It can be negative on test data.

Lower is better for MAE and RMSE; higher is better for $R^2$.

### 6.3 Train/test split

```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
print("Train:", X_train.shape[0], "| Test:", X_test.shape[0])
```

The test set is **untouched** until model selection is finished.

### 6.4 Helper + OLS baseline

```python
def evaluate(name, y_true, y_pred, **extra):
    return {
        "Model": name, **extra,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }

linear_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])
linear_pipeline.fit(X_train, y_train)

linear_results = evaluate("Linear Regression", y_test, linear_pipeline.predict(X_test))
pd.DataFrame([linear_results])
```

With an intercept, OLS predictions don't change with feature scaling, but the coefficient values do. We scale here so coefficients are comparable with Ridge later.

### 6.5 Ridge with default alpha

```python
ridge_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])
ridge_pipeline.fit(X_train, y_train)

ridge_results = evaluate("Ridge (alpha=1)", y_test, ridge_pipeline.predict(X_test))
pd.DataFrame([linear_results, ridge_results])
```

Ridge won't beat OLS on every dataset. It helps only when the reduction in variance outweighs the added bias.

### 6.6 Tune alpha with cross-validation

Search on a **log scale**, because alpha's effect is multiplicative across orders of magnitude.

```python
alpha_values = np.logspace(-3, 3, 50)

search_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge())
])

cv = KFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=search_pipeline,
    param_grid={"model__alpha": alpha_values},
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
    return_train_score=True
)
grid_search.fit(X_train, y_train)

print("Best alpha:", grid_search.best_params_["model__alpha"])
print("Best CV RMSE:", -grid_search.best_score_)
```

**Why negative?** sklearn always maximises scores, so error metrics are negated. Flip the sign to get RMSE.

### 6.7 Train vs CV error across alpha

```python
cv_results = pd.DataFrame(grid_search.cv_results_)
alphas = cv_results["param_model__alpha"].astype(float)

plt.figure(figsize=(9, 5))
plt.semilogx(alphas, -cv_results["mean_train_score"], label="Train RMSE")
plt.semilogx(alphas, -cv_results["mean_test_score"], label="CV RMSE")
plt.axvline(grid_search.best_params_["model__alpha"], linestyle="--", color="grey", label="Best alpha")
plt.xlabel("alpha (log scale)")
plt.ylabel("RMSE")
plt.title("Ridge: train vs CV RMSE")
plt.legend()
plt.grid(True)
plt.show()
```

**How to read this plot:**
- Training error always rises as alpha grows, because the model is increasingly constrained.
- CV error is high at large alpha (underfitting). It may also be high at tiny alpha (overfitting).
- A large gap between train and CV error indicates variance (overfitting).
- If the CV curve is flat near its minimum, don't over-interpret tiny differences. Choosing the larger (simpler) alpha in that flat region is a reasonable choice.

### 6.8 Coefficient path

```python
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)

alphas_path = np.logspace(-3, 5, 100)
coefs = np.array([Ridge(alpha=a).fit(X_train_scaled, y_train).coef_ for a in alphas_path])

plt.figure(figsize=(9, 5))
plt.semilogx(alphas_path, coefs)
plt.axhline(0, color="black", linewidth=0.8)
plt.xlabel("alpha (log scale)")
plt.ylabel("Coefficient (standardised)")
plt.title("Ridge coefficient paths")
plt.legend(X.columns, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()
```

All coefficients shrink smoothly towards 0 but never hit it exactly. Some coefficients can even change sign along the way when features are correlated.

### 6.9 Faster alternative: `RidgeCV`

```python
ridgecv_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RidgeCV(alphas=alpha_values))   # cv=None -> efficient leave-one-out
])
ridgecv_pipeline.fit(X_train, y_train)
print("RidgeCV alpha:", ridgecv_pipeline.named_steps["model"].alpha_)
```

The leave-one-out happens inside `RidgeCV`, after the scaler has already been fitted on all of `X_train`. This leaks slightly. Use `GridSearchCV` when you need fully leakage-safe tuning.

### 6.10 Final test evaluation

```python
best_ridge = grid_search.best_estimator_
best_results = evaluate(
    "Tuned Ridge", y_test, best_ridge.predict(X_test),
    Alpha=grid_search.best_params_["model__alpha"]
)
pd.DataFrame([linear_results, ridge_results, best_results])
```

This is the **one** held-out evaluation. Report these numbers as the estimate of performance on unseen data.

### 6.11 Compare coefficients

```python
comparison = pd.DataFrame({
    "Feature": X.columns,
    "OLS": linear_pipeline.named_steps["model"].coef_,
    "Ridge": best_ridge.named_steps["model"].coef_,
})
comparison.reindex(comparison["Ridge"].abs().sort_values(ascending=False).index)
```

**Interpretation:** each coefficient is the change in predicted $y$ for a **one-standard-deviation** increase in that feature, holding the others fixed.
- Shrinkage is not proportional across features.
- Correlated features (e.g. `s1` and `s2` in this dataset) often show the largest change from OLS. OLS may give them large opposite-sign weights, and Ridge tames them.
- A small coefficient does **not** prove a feature is unimportant.

### 6.12 Actual vs predicted

```python
best_predictions = best_ridge.predict(X_test)

plt.figure(figsize=(6, 6))
plt.scatter(y_test, best_predictions, alpha=0.7)
lims = [min(y_test.min(), best_predictions.min()), max(y_test.max(), best_predictions.max())]
plt.plot(lims, lims, linestyle="--")
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("Tuned Ridge: actual vs predicted")
plt.grid(True)
plt.show()
```

Points near the diagonal are good predictions. Systematic deviations, such as over-predicting low values and under-predicting high values, reveal bias. That pattern is typical of shrinkage.

---

## 7. The maths

For centred $X$ and $y$:

$$
\hat{\boldsymbol{\beta}}_{\text{ridge}} = (X^\top X + \alpha I)^{-1} X^\top y
\qquad \text{vs} \qquad
\hat{\boldsymbol{\beta}}_{\text{OLS}} = (X^\top X)^{-1} X^\top y
$$

$X^\top X$ can be singular or near-singular (because of collinearity or $p > n$). Adding $\alpha I$ with $\alpha > 0$ makes the matrix **always invertible** and well-conditioned. In practice, solvers never compute the inverse explicitly.

### SVD view: what Ridge actually shrinks

With $X = U D V^\top$ and singular values $d_j$:

$$
\hat{y}_{\text{ridge}} = \sum_{j=1}^{p} \mathbf{u}_j \, \frac{d_j^2}{d_j^2 + \alpha} \, \mathbf{u}_j^\top y
$$

- OLS uses a shrink factor of 1 for every direction.
- Ridge shrinks each principal direction by $\frac{d_j^2}{d_j^2 + \alpha}$.
- **Low-variance directions** (small $d_j$, which is where collinearity lives) are shrunk the most.

**Effective degrees of freedom:**

$$
\text{df}(\alpha) = \sum_{j=1}^{p} \frac{d_j^2}{d_j^2 + \alpha}
$$

This goes from $p$ (when $\alpha = 0$) down to $0$ (as $\alpha \to \infty$).

### Bayesian view

Ridge is the **MAP estimate** under a Gaussian likelihood combined with a Gaussian prior $\beta_j \sim \mathcal{N}(0, \tau^2)$, where $\alpha = \sigma^2 / \tau^2$. A stronger prior belief that coefficients are small corresponds to a larger $\alpha$.

### Gradient view

The gradient of the penalty is $2\alpha\beta$, which pulls every weight towards 0 at each step. This is exactly **weight decay** in neural networks.

---

## 8. Assumptions and limitations

1. **Still linear.** Nonlinear patterns need feature engineering, `KernelRidge`, or nonlinear models.
2. **No feature elimination.** All features stay in the model.
3. **Scale-dependent.** Always standardise.
4. **Alpha must be tuned.** A bad alpha can cause either overfitting or underfitting.
5. **Coefficients are biased.** They are not causal effects, and standard OLS p-values don't apply.
6. **Outlier-sensitive.** Squared loss is sensitive to extreme points; consider `HuberRegressor` if outliers are a problem.

---

## 9. Interview questions

**Q1. What is Ridge Regression, in one line?**
OLS plus an L2 penalty $\alpha \sum_j \beta_j^2$. It shrinks coefficients to reduce variance, trading a little bias for better generalisation.

**Q2. What does alpha do, and how do you choose it?**
It sets the shrinkage strength. $\alpha \to 0$ gives OLS; $\alpha \to \infty$ gives predictions equal to the mean of $y$. Choose it with cross-validation over a log-spaced grid, inside a pipeline with scaling, then evaluate once on the test set.

**Q3. Does Ridge do feature selection?**
No. Coefficients approach 0 but don't reach it. Use Lasso or Elastic Net for sparsity.

**Q4. Why scale features first?**
The penalty acts on coefficient magnitudes, which depend on feature units. Without scaling, features are penalised unequally for arbitrary reasons.

**Q5. Why use a Pipeline?**
So the scaler is fitted only on each training fold during CV. This prevents leakage and applies identical preprocessing at prediction time.

**Q6. How does Ridge handle multicollinearity?**
Adding $\alpha I$ to $X^\top X$ fixes ill-conditioning. In SVD terms, Ridge shrinks low-variance directions the most. Correlated features end up sharing weight roughly equally instead of taking huge opposite-sign values. Note that the correlation itself still exists.

**Q7. Ridge vs Lasso?**
L2 vs L1. Ridge keeps all features and spreads weight across correlated groups; Lasso zeros some features out and tends to pick one feature from a correlated group arbitrarily. Ridge has a closed-form solution; Lasso doesn't. Geometrically: circle vs diamond.

**Q8. When would you choose Elastic Net?**
When you want sparsity, but features come in correlated groups where Lasso alone is unstable.

**Q9. Why isn't the intercept penalised?**
Penalising it would make the model depend on the arbitrary location of the target. The intercept just captures the mean level of $y$.

**Q10. Is Ridge biased? Why use a biased estimator?**
Yes, it is biased towards 0. Expected test error is $\text{bias}^2 + \text{variance} + \text{noise}$, and a small increase in bias can buy a large reduction in variance.

**Q11. What happens when $p > n$?**
$X^\top X$ is singular, so OLS has infinitely many solutions. Ridge still has a unique solution for any $\alpha > 0$.

**Q12. Can Ridge be used for classification?**
Yes, with `RidgeClassifier`, which encodes classes as $\pm 1$ and fits a Ridge regression. More commonly, people use `LogisticRegression` with an L2 penalty. That is the default there, and note that `C` $= 1/\alpha$ is the *inverse* of regularisation strength.

**Q13. Is the same alpha comparable between `Ridge` and `Lasso` in sklearn?**
No. Lasso's loss is scaled by $\frac{1}{2n}$ and Ridge's isn't, so tune each one separately.

**Q14. Bayesian interpretation?**
Ridge is the MAP estimate with a Gaussian prior on the coefficients, where $\alpha = \sigma^2 / \tau^2$. Lasso corresponds to a Laplace prior.

**Q15. How do you interpret Ridge coefficients?**
On standardised features, a coefficient is the change in prediction per one-SD increase in that feature, holding others fixed. The values are shrunk, so treat them as predictive weights, not causal effects. For importance, use permutation importance instead.

**Q16. How do you handle categorical features?**
One-hot encode them, using a `ColumnTransformer` so numeric and categorical columns get different preprocessing. With Ridge you don't need to drop a reference category, because the penalty resolves the dummy-variable collinearity. Whether to scale the dummies is a judgement call, since scaling changes how strongly each one is penalised.

**Q17. Does the target scale matter?**
Yes. The RSS term scales with $y^2$, so the optimal $\alpha$ changes if you rescale $y$. If you transform the target (e.g. with a log), use `TransformedTargetRegressor` and report metrics on the original scale.

**Q18. Is Ridge affected by outliers?**
Yes, because of the squared loss. Use `HuberRegressor` or another robust method if outliers are a concern.

**Q19. Does Ridge always beat OLS?**
No. With plenty of data, few features, and low collinearity, OLS may generalise just as well.

**Q20. How does Ridge relate to neural networks?**
The L2 penalty is equivalent to weight decay. In plain SGD, each update multiplies the weights by $(1 - \eta \cdot 2\alpha)$.

---

## 10. Scenario questions

**S1. The model is great on training data and poor on test data.**
- Check for overfitting, leakage, and whether the train and test sets are representative.
- Fit a scaled Ridge model, tune alpha with CV, and compare held-out performance.
- Plot train vs CV error to confirm it is a variance problem.

**S2. You have 100 features and many are highly correlated. Ridge or Lasso?**
- Ridge is a strong default if many features carry signal and coefficient stability matters.
- Try Lasso or Elastic Net if you need a sparse, interpretable model.
- Compare all of them on the same CV folds.

**S3. CV error rises steadily as alpha increases.**
- You are moving into underfitting, so the useful range is smaller alpha.
- Extend the grid downwards.
- If OLS is best, regularisation simply isn't needed here.

**S4. A colleague says a feature is unimportant because its Ridge coefficient is near zero.**
That conclusion isn't justified on its own:
- Correlated features share weight.
- Coefficient size depends on scaling and alpha.
- Use permutation importance and domain knowledge instead.

**S5. CV results are excellent but test performance is poor.**
- There may be distribution shift between the data and the test set.
- The split may not match deployment conditions (for example, using a random split on time-series data instead of `TimeSeriesSplit`).
- With a small dataset, you may have overfit the CV process itself. Use nested CV.

**S6. You have 10 million rows and a sparse feature matrix.**
- Use `solver='sag'`, `'saga'`, or `'lsqr'` with scaled features, or switch to `SGDRegressor(penalty='l2')`.
- Avoid `'svd'` and `'cholesky'` on huge or sparse data.

**S7. Coefficients must be non-negative (e.g. a marketing-mix model).**
Use `Ridge(positive=True)`, which requires `solver='lbfgs'` (or `'auto'`, which picks it).

---

## 11. Common pitfalls checklist

- [ ] Fitting the scaler on all data before splitting (leakage)
- [ ] Using the test set to pick alpha
- [ ] Using a linear alpha grid instead of a log-spaced one
- [ ] Comparing coefficient sizes across unscaled features
- [ ] Reading shrunk coefficients as causal effects
- [ ] Reusing a Lasso alpha for Ridge, or vice versa
- [ ] Using a random K-fold split on time-series data

---

## 12. Quick revision sheet

| Concept | Remember |
|---|---|
| Ridge | OLS + L2 penalty $\alpha \sum_j \beta_j^2$ |
| Closed form | $(X^\top X + \alpha I)^{-1} X^\top y$ |
| Small $\alpha$ | Approaches OLS |
| Large $\alpha$ | Coefficients $\to 0$; underfitting |
| Zero coefficients | Never exactly zero |
| Shrinks most | Low-variance (collinear) directions |
| Scaling | Essential, inside a Pipeline |
| Tuning | Log grid + CV (`GridSearchCV` / `RidgeCV`) |
| Bayesian | Gaussian prior |
| Trade-off | Small increase in bias, larger reduction in variance |

**30-second interview answer:**
Ridge is linear regression with an L2 penalty on the coefficients. It shrinks weights to reduce variance, which helps especially with correlated features or many predictors, at the cost of a little bias. Alpha controls the strength. Because the penalty depends on coefficient magnitudes, I standardise features and tune alpha with cross-validation inside a pipeline, then evaluate once on a held-out test set. Unlike Lasso, it doesn't zero out features.

---